# Step 4. The language step: how alike the measures are worded, and the map

The Map view places every measure so that measures **worded alike sit close together**, whatever their category. The
tooltips also name each measure's closest relatives in earlier budgets and say how new its wording is. All of it comes
from the words of the measures, in five steps:

1. **Count the words that matter (TF-IDF).** Each measure becomes a list of scores, one per word or two-word phrase.
   A phrase scores high when it is frequent in this measure and rare across all measures ("small modular reactors"),
   low when every measure uses it ("budget 2025 proposes"). Common English words are dropped.
2. **Compress (SVD).** About 23,000 phrases are folded into 100 themes, so that measures using different words for
   the same thing ("child care", "early learning") still come out alike.
3. **Compare (cosine similarity).** Two measures' theme scores are compared: 1 means worded the same, 0 nothing in
   common.
4. **Draw the map (t-SNE).** The measures are laid out on a flat map so that each one's closest relatives stay
   near it. Only closeness matters: the axes mean nothing, and distances between far-apart groups mean little.
5. **Per measure and category:** its three closest measures in **earlier** budgets, its **novelty** (1 minus the
   closest similarity: high means nothing earlier was worded like it), the category most of its closest neighbours
   belong to when that is not its own (`like`), and each category's **anchor** (the spot where its measures are
   densest on the map).

**Same results on every run.** Each random step has a fixed seed (7), the computer's processor cores are limited to
one (t-SNE adds up its numbers in a different order on different machines otherwise), and `requirements.txt` pins the
library versions. Running this notebook again gives the same map.

**Budgets 2022 to 2026 only.** Budget 2021 is left off the page: its measures are not on the map and are nobody's
closest earlier measure (it still counts in step 5, in the Priorities averages of 2022 and 2023).

Inputs: `data/measures.csv` (the text), `data/categorised.csv` (the category). Outputs: `data/language.csv` (one row
per measure) and `data/category_anchors.csv`.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"          # one core: t-SNE gives the same numbers on any machine (set before numpy)
from collections import Counter

import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize

DATA = "data"
SEED = 7                                   # the seed of every random step
N_THEMES = 100                             # SVD: number of themes
PERPLEXITY = 30                            # t-SNE: about how many neighbours each measure keeps close
N_NEAR = 3                                 # closest earlier measures listed per measure
N_LIKE, LIKE_MIN = 8, 4                    # `like`: among the 8 closest measures, at least 4 in another category
YEARS = [2022, 2023, 2024, 2025, 2026]     # the budgets on the page and the map (Budget 2021 is left out)
GRID, BW = 200, (0.028, 0.045)             # anchors: a 200 x 200 grid; smoothing as on the page (map 0.62 times as tall as wide)
print("numpy", np.__version__, "| scipy", scipy.__version__, "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

numpy 2.0.2 | scipy 1.13.1 | scikit-learn 1.6.1 | pandas 2.3.3


## The documents
One document per measure: its heading, twice (the heading says what the measure is about, so it weighs more), then
its text. Measures that exist only in a cost table have their one-line description as text.

In [2]:
text = pd.read_csv(f"{DATA}/measures.csv", usecols=["id", "text"])
cat = pd.read_csv(f"{DATA}/categorised.csv")
M = cat.merge(text, on="id", how="left", validate="one_to_one")    # same order as step 2 (year, chapter, page order)
assert (M["id"].values == cat["id"].values).all()
M = M[M["year"].isin(YEARS)].reset_index(drop=True)              # the budgets on the map only
print(f"Budgets {YEARS[0]} to {YEARS[-1]}: {len(M):,} measures (left out: {len(cat) - len(M):,} from other budgets)")
M["text"] = M["text"].fillna("")
docs = ((M["heading"] + ". ") * 2 + M["text"]).tolist()
n = len(docs)
print(f"{n:,} documents, {sum(len(d.split()) for d in docs):,} words in all")

Budgets 2022 to 2026: 999 measures (left out: 318 from other budgets)
999 documents, 229,157 words in all


## 1. TF-IDF: the words that matter in each measure
Settings: English stop words dropped; single words and two-word phrases; a phrase must appear in at least 2 measures
and in no more than 40% of them; repeated words count less than proportionally (`sublinear_tf`); only words made of
letters (no numbers, no dollar amounts).

Example below: the phrases that weigh most in Budget 2026's Productivity Mega Deduction.

In [3]:
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, max_df=0.4, sublinear_tf=True,
                        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z-]+\b")
X = tfidf.fit_transform(docs)                                      # measures x phrases, mostly zeros
terms = np.array(tfidf.get_feature_names_out())
print(f"{X.shape[1]:,} words and phrases kept")

EXAMPLE = M.index[M["heading"].str.startswith("Kickstarting Canada's Investment Supercycle")][0]
row = X[EXAMPLE].toarray().ravel()
print(f"\nExample: {M.loc[EXAMPLE, 'heading']} ({M.loc[EXAMPLE, 'year']})")
for j in np.argsort(-row)[:12]:
    print(f"   {row[j]:.3f}  {terms[j]}")

18,332 words and phrases kept

Example: Kickstarting Canada's Investment Supercycle with the Productivity Mega Deduction (2026)
   0.161  productivity mega
   0.161  mega deduction
   0.161  mega
   0.154  deduction
   0.136  depreciation
   0.130  generally
   0.130  productivity
   0.129  equipment
   0.124  bonus
   0.115  canada investment
   0.113  assets
   0.107  vessels


## 2. SVD: from phrases to 100 themes
Each measure becomes 100 numbers. Each theme is a set of phrases that tend to appear together; below, the phrases that
define the first five themes (a theme can mix a positive and a negative side).

In [4]:
svd = TruncatedSVD(n_components=N_THEMES, random_state=SEED)
Z = normalize(svd.fit_transform(X))                                # length 1, so a dot product is the cosine
print(f"the 100 themes keep {svd.explained_variance_ratio_.sum():.0%} of the variation in the phrase scores")
for k in range(5):
    top = terms[np.argsort(-svd.components_[k])[:6]]
    print(f"   theme {k + 1}: {', '.join(top)}")

the 100 themes keep 26% of the variation in the phrase scores
   theme 1: new, canadians, canadian, million years, communities, program
   theme 2: indigenous, communities, health, nations, services, care
   theme 3: tax, canadians, income, care, housing, financial
   theme 4: funding proposed, proposed, security, financial, service, information
   theme 5: funding proposed, proposed, tax, tax credit, indigenous, credit


## 3. Similarity: how alike two measures are worded
Cosine similarity of the theme scores, from 0 (nothing in common) to 1 (worded the same). Example: the measures
worded most like the Mega Deduction, in any budget.

In [5]:
sim = Z @ Z.T
np.fill_diagonal(sim, -np.inf)                                     # a measure is not its own neighbour
order = np.argsort(-sim, axis=1, kind="stable")                    # each measure's neighbours, closest first
for j in order[EXAMPLE, :6]:
    print(f"   {sim[EXAMPLE, j]:.2f}  {M.loc[j, 'year']}  {M.loc[j, 'heading'][:90]}")

   0.89  2025  Productivity Super-Deduction
   0.71  2024  Incentivizing More Innovation and Productivity
   0.61  2023  Enhancing the Reduced Tax Rates for Zero-Emission Technology Manufacturers
   0.60  2026  Making it easier to do business
   0.59  2026  Backing Canada's next generation of entrepreneurs
   0.59  2022  Cutting Taxes for Canada's Growing Small Businesses


## 4. The map (t-SNE)
t-SNE starts from the two main directions of the theme scores (`init="pca"`, so the start is not random either) and
moves the measures until each one's neighbours on the map are, as far as possible, its closest measures in wording.
Settings: perplexity 30, cosine distance, seed 7; positions are then stretched to fill 0 to 1 on both axes.

In [6]:
tsne = TSNE(n_components=2, perplexity=PERPLEXITY, init="pca", metric="cosine", random_state=SEED)
xy = tsne.fit_transform(Z)
xy = (xy - xy.min(0)) / (xy.max(0) - xy.min(0))                    # 0 to 1 on both axes
M["x"], M["y"] = xy[:, 0].round(4), xy[:, 1].round(4)
print(f"map drawn: {tsne.n_iter_} iterations, final KL divergence {tsne.kl_divergence_:.3f} (lower = neighbours better kept)")

# how well the map keeps neighbours: of each measure's 10 closest measures in wording, how many are among its 10 closest on the map
d2 = ((xy[:, None, :] - xy[None, :, :]) ** 2).sum(-1)
np.fill_diagonal(d2, np.inf)
on_map = np.argsort(d2, axis=1, kind="stable")[:, :10]
kept = np.mean([len(set(order[i, :10]) & set(on_map[i])) / 10 for i in range(n)])
print(f"on average {kept:.0%} of a measure's 10 closest measures in wording are among its 10 closest dots on the map")

map drawn: 999 iterations, final KL divergence 1.224 (lower = neighbours better kept)
on average 62% of a measure's 10 closest measures in wording are among its 10 closest dots on the map


## 5. Per measure: closest earlier measures, novelty, and `like`
- `near1` to `near3`: the three measures from **earlier budgets** worded most like it, with their similarity
  (Budget 2022 has none: it is the first on the map);
- `novelty`: 1 minus the closest of those similarities (0.2: an earlier measure was worded almost the same; 0.8:
  nothing earlier was);
- `like`: among its 8 closest measures in any budget, the category that at least 4 of them share, when it is not
  the measure's own (a measure filed under Affordability but worded like Clean energy measures, for example).

In [7]:
years = M["year"].values
cats = M["category"].values
near_cols = {f"near{k + 1}": [] for k in range(N_NEAR)} | {f"near{k + 1}_sim": [] for k in range(N_NEAR)}
novelty, like = [], []
for i in range(n):
    earlier = [j for j in order[i] if years[j] < years[i]][:N_NEAR]   # closest first, earlier budgets only
    for k in range(N_NEAR):
        near_cols[f"near{k + 1}"].append(M.loc[earlier[k], "id"] if k < len(earlier) else "")
        near_cols[f"near{k + 1}_sim"].append(round(float(sim[i, earlier[k]]), 2) if k < len(earlier) else np.nan)
    novelty.append(round(max(0.0, 1 - float(sim[i, earlier[0]])), 2) if earlier else np.nan)   # never below 0 (rounding)
    c, cnt = Counter(cats[order[i, :N_LIKE]]).most_common(1)[0]        # most common category among the 8 closest
    like.append(c if (c != cats[i] and cnt >= LIKE_MIN) else "")
for k, v in near_cols.items():
    M[k] = v
M["novelty"], M["like"] = novelty, like
print(f"novelty for {M['novelty'].notna().sum():,} measures (all but Budget {years.min()}'s); "
      f"{(M['like'] != '').sum()} measures worded mostly like another category")

print("\nMost novel measures in each budget (nothing earlier worded like them):")
for year, part in M[M["year"] > years.min()].groupby("year"):
    for _, m in part.nlargest(3, "novelty").iterrows():
        print(f"   {year}  {m['novelty']:.2f}  {m['heading'][:90]}")
print("\nMost repeated (worded almost like an earlier measure):")
for _, m in M.nsmallest(6, "novelty").iterrows():
    first = M.loc[M["id"] == m["near1"], ["year", "heading"]].iloc[0]
    print(f"   {m['year']}  {m['novelty']:.2f}  {m['heading'][:60]}  <- {first['year']} {first['heading'][:50]}")

novelty for 796 measures (all but Budget 2022's); 135 measures worded mostly like another category

Most novel measures in each budget (nothing earlier worded like them):
   2023  0.81  GAC Adjustments for Non-Discretionary Cost Fluctuations
   2023  0.75  Reimbursing Partners for Costs Related to Illegal Blockades and Occupations in 2022
   2023  0.75  Renewing Pay System Resources
   2024  0.69  Canada's standards-setting and accreditation system
   2024  0.62  Terry Fox Humanitarian Award
   2024  0.61  Supporting the Canadian Chamber of Commerce's Business Data Lab
   2025  0.55  Ensuring Access to Financing
   2025  0.55  Protecting Against Economic Abuse
   2025  0.51  Supporting Medical School Infrastructure
   2026  0.52  Drawing talent from across the economy into government
   2026  0.51  Mackenzie Valley Highway: connecting communities and protecting sovereignty
   2026  0.50  Archiving the records of former Prime Minister Trudeau

Most repeated (worded almost like an earlie

## How alike each category's measures are worded
For each category: on average, how many of a measure's 8 closest measures share its category. A category whose
measures share their wording (Indigenous, Defence) forms one area on the map; one defined by purpose across many
subjects (Government, Other) is spread out. For reference, picking 8 measures at random would give about 1 in 15.

In [8]:
same = np.array([(cats[order[i, :N_LIKE]] == cats[i]).mean() for i in range(n)])
check = pd.DataFrame({"category": cats, "same": same}).groupby("category")["same"].agg(["count", "mean"])
check.columns = ["measures", "share of 8 closest in the same category"]
print(check.sort_values(check.columns[1], ascending=False).round(2).to_string())

                                   measures  share of 8 closest in the same category
category                                                                            
Housing                                  69                                     0.78
Culture & communities                    81                                     0.74
Indigenous                               89                                     0.71
Research, AI & technology                40                                     0.68
Clean energy & climate                   74                                     0.67
Jobs, skills & immigration               95                                     0.66
Taxation                                 46                                     0.65
Defence & security                      126                                     0.64
Nature & environment                     29                                     0.62
Government                               78                      

## Category anchors
For each category, the spot on the map where its measures are densest: each measure spreads a small bump around its
dot (the same smoothing as the page's shaded areas), the bumps are added up on a 200 by 200 grid, and the highest
point is the anchor. The Gravity view's earlier versions used the anchors to place a budget's centre of gravity; the
page also uses them to place category names.

In [9]:
gx, gy = np.meshgrid(np.linspace(0, 1, GRID), np.linspace(0, 1, GRID))
anchors = []
for k, name in enumerate(pd.read_csv(f"{DATA}/category_list.csv")["category"]):
    p = xy[cats == name]
    dens = np.exp(-(((gx[..., None] - p[:, 0]) / BW[0]) ** 2 + ((gy[..., None] - p[:, 1]) / BW[1]) ** 2) / 2).sum(-1)
    iy, ix = np.unravel_index(np.argmax(dens), dens.shape)
    anchors.append({"cat_id": k, "category": name, "measures": len(p),
                    "ax": round(float(gx[iy, ix]), 4), "ay": round(float(gy[iy, ix]), 4)})
anchors = pd.DataFrame(anchors)
print(anchors.to_string(index=False))

 cat_id                          category  measures     ax     ay
      0                           Housing        69 0.6281 0.6633
      1    Affordability & social support        59 0.9146 0.3970
      2                       Health care        54 0.3116 0.7236
      3 Trade, industry & competitiveness       113 0.6131 0.3568
      4        Jobs, skills & immigration        95 0.5075 0.8945
      5                    Infrastructure        33 0.5327 0.0302
      6         Research, AI & technology        40 0.6834 0.4975
      7             Culture & communities        81 0.0804 0.3417
      8              Nature & environment        29 0.5327 0.1357
      9            Clean energy & climate        74 0.7538 0.3015
     10                          Taxation        46 0.9045 0.6884
     11                        Government        78 0.4774 0.6332
     12                Defence & security       126 0.5578 0.4874
     13                        Indigenous        89 0.4221 0.4020
     14   

## Save

In [10]:
cols = ["id", "year", "heading", "category", "x", "y", "novelty"] + list(near_cols) + ["like"]
M[cols].to_csv(f"{DATA}/language.csv", index=False)
anchors.to_csv(f"{DATA}/category_anchors.csv", index=False)
print(f"wrote {DATA}/language.csv ({len(M):,} rows) and {DATA}/category_anchors.csv ({len(anchors)} rows)")

wrote data/language.csv (999 rows) and data/category_anchors.csv (15 rows)
